# Emotion Classification with Attention Mechanisms
### Multi-Class Text Emotion Detection across 6 Core Emotions

**Objective:**
Build and benchmark sequence models with attention mechanisms against a fine-tuned transformer (DistilBERT) for multi-class emotion classification across **6 core emotions**:
1. **Joy**
2. **Sadness**
3. **Anger**
4. **Fear**
5. **Surprise**
6. **Disgust**

---
### Architectures Compared:
1. **LSTM** (with pre-trained GloVe 100d embeddings)
2. **GRU** (with pre-trained GloVe 100d embeddings)
3. **BiLSTM + Attention** (with pre-trained GloVe 100d embeddings and Bahdanau attention weight extraction)
4. **Fine-Tuned DistilBERT** (Hugging Face Trainer API)

---
### Interpretability:
We extract dynamic attention weights from the BiLSTM + Attention model to generate **token-level attention heatmaps** explaining the exact words driving each emotional classification.

## 1. Environment Setup & Imports

In [ ]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

from src.dataset import (
    EMOTIONS,
    EMOTION_TO_ID,
    ID_TO_EMOTION,
    clean_text,
    prepare_goemotions_df,
    Vocabulary,
    load_glove_matrix,
    EmotionDataset
)
from src.models import LSTMClassifier, GRUClassifier, BiLSTMAttentionClassifier
from src.train import train_model
from src.evaluate import evaluate_pytorch_model
from src.visualize import plot_attention_heatmap, plot_confusion_matrix, plot_model_comparisons

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device in use: {device}')

## 2. Load & Explore GoEmotions Dataset (Ekman 6 Core Mapping)

The GoEmotions dataset (Google Research) consists of Reddit comments annotated with 27 emotions + neutral.
We map these 28 classes into **Paul Ekman's 6 basic core emotions**:
- **Joy:** *admiration, amusement, approval, excitement, gratitude, joy, love, optimism, pride, relief*
- **Sadness:** *disappointment, embarrassment, grief, remorse, sadness*
- **Anger:** *anger, annoyance, disapproval*
- **Fear:** *fear, nervousness*
- **Surprise:** *confusion, curiosity, realization, surprise*
- **Disgust:** *disgust*

In [ ]:
train_df = prepare_goemotions_df('Simplified/train-00000-of-00001.parquet', use_ekman=True)
val_df = prepare_goemotions_df('Simplified/validation-00000-of-00001.parquet', use_ekman=True)
test_df = prepare_goemotions_df('Simplified/test-00000-of-00001.parquet', use_ekman=True)

print(f'Train samples: {len(train_df):,}')
print(f'Validation samples: {len(val_df):,}')
print(f'Test samples: {len(test_df):,}')

train_df.head(10)

In [ ]:
# Visualize Emotion Class Distribution
plt.figure(figsize=(8, 4), dpi=120)
order = train_df['emotion'].value_counts().index
sns.countplot(data=train_df, x='emotion', order=order, palette='viridis')
plt.title('Emotion Distribution in Training Set (Mapped Ekman Classes)', fontsize=12, fontweight='bold')
plt.xlabel('Emotion Class')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Tokenization & Pre-Trained GloVe Word Embeddings

We tokenize the cleaned texts, build a vocabulary with special tokens `<pad>` (index 0) and `<unk>` (index 1), and load **GloVe 100-dimensional pre-trained embeddings** to initialize the embedding layers.

In [ ]:
vocab = Vocabulary()
vocab.build_vocab(train_df['clean_text'].tolist(), min_freq=2)
print(f'Vocabulary size: {len(vocab):,} unique tokens')

# Load GloVe 100d weights matrix
embedding_matrix = load_glove_matrix(vocab, embedding_dim=100)
print(f'Embedding matrix shape: {embedding_matrix.shape}')

In [ ]:
# PyTorch DataLoaders
max_len = 50
batch_size = 64

train_dataset = EmotionDataset(train_df['clean_text'], train_df['label'], vocab, max_len=max_len)
val_dataset = EmotionDataset(val_df['clean_text'], val_df['label'], vocab, max_len=max_len)
test_dataset = EmotionDataset(test_df['clean_text'], test_df['label'], vocab, max_len=max_len)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size * 2, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size * 2, shuffle=False)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}')

## 4. Sequence Model Architectures with Attention

### Mathematical Formulation of Bahdanau Attention:
Given hidden states $H = [h_1, h_2, \dots, h_T] \in \mathbb{R}^{T \times 2D}$ from the Bidirectional LSTM:
1. **Energy Scoring:** $u_t = \tanh(W_a h_t + b_a)$ where $W_a \in \mathbb{R}^{A \times 2D}$
2. **Attention Weights:** $\alpha_t = \frac{\exp(v_a^\top u_t)}{\sum_{k=1}^T \exp(v_a^\top u_k)}$ with padding masked
3. **Context Vector:** $c = \sum_{t=1}^T \alpha_t h_t \in \mathbb{R}^{2D}$
4. **Classification:** $\hat{y} = \text{softmax}(W_c c + b_c)$

In [ ]:
# Inspect model architecture
bilstm_sample = BiLSTMAttentionClassifier(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_dim=128,
    num_layers=2,
    num_classes=6,
    attention_dim=128,
    weights_matrix=embedding_matrix
)
print(bilstm_sample)

## 5. Model Benchmark & Comparative Evaluation

We evaluate all four architectures on the unseen test set:
- **LSTM**
- **GRU**
- **BiLSTM + Attention**
- **Fine-Tuned DistilBERT**

In [ ]:
if os.path.exists('results/model_comparison.csv'):
    comparison_df = pd.read_csv('results/model_comparison.csv')
    print('Architecture Comparison:')
    print(comparison_df.to_string(index=False))
    
    # Plot performance comparison
    plot_model_comparisons(comparison_df)
    plt.show()
else:
    print('Run train_all_and_compare.py to generate full comparison benchmark.')

In [ ]:
# Display Confusion Matrices
from PIL import Image

fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=130)
model_cm_files = [
    ('LSTM', 'results/confusion_matrix_lstm.png'),
    ('GRU', 'results/confusion_matrix_gru.png'),
    ('BiLSTM + Attention', 'results/confusion_matrix_bilstm___attention.png'),
    ('Fine-Tuned DistilBERT', 'results/confusion_matrix_fine_tuned_distilbert.png')
]

for idx, (title, path) in enumerate(model_cm_files):
    ax = axes[idx // 2, idx % 2]
    if os.path.exists(path):
        img = Image.open(path)
        ax.imshow(img)
        ax.axis('off')
    else:
        ax.text(0.5, 0.5, f'{title}\n(Check results folder)', ha='center', va='center')
        ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Attention Heatmap Visualizations & Token Interpretability

Here we visualize the token-level attention distribution for sample sentences across each emotion class.
Notice how the model concentrates attention weight directly on emotional anchor words.

In [ ]:
# Load trained BiLSTM + Attention model
attn_model = BiLSTMAttentionClassifier(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_dim=128,
    num_layers=2,
    num_classes=6
)
if os.path.exists('saved_models/bilstm_attention.pt'):
    attn_model.load_state_dict(torch.load('saved_models/bilstm_attention.pt', map_location=device))
    attn_model.to(device)
    attn_model.eval()

def predict_and_explain(text):
    clean = clean_text(text)
    ids, length, tokens = vocab.encode(clean, max_len=50)
    inp = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        logits, weights = attn_model(inp, return_attention=True)
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        pred_id = int(np.argmax(probs))
        pred_emo = ID_TO_EMOTION[pred_id]
        conf = float(probs[pred_id])
        token_weights = weights[0].cpu().numpy()[:length]
    
    fig = plot_attention_heatmap(tokens, token_weights, pred_label=pred_emo, confidence=conf)
    plt.show()
    return pred_emo, conf

# Test samples
predict_and_explain('I am so overjoyed and thrilled by this fantastic news!')
predict_and_explain('I feel completely empty and broken hearted.')
predict_and_explain('This is totally infuriating and ridiculous.')
predict_and_explain('I am terrified of being abandoned alone.')
predict_and_explain('Wait what? I was completely shocked and never expected that.')
predict_and_explain('The rotting mold on that bread was utterly repulsive and sickening.')

## 7. Interactive Custom Inference

Try entering any sentence to predict the dominant emotion and view confidence across all 6 core emotions:

In [ ]:
custom_sentence = 'I love working on deep learning projects, it is so much fun!'
pred, conf = predict_and_explain(custom_sentence)
print(f"Input: '{custom_sentence}'")
print(f'Dominant Emotion: {pred.upper()} ({conf:.1%})')